In [19]:


from pathlib import Path
from urllib.request import urlretrieve
import json
import re

import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split



CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"

for directory in [RAW_DATA_DIR, PROCESSED_DATA_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("项目根目录：", PROJECT_ROOT.resolve())
print("原始数据目录：", RAW_DATA_DIR.resolve())
print("处理后数据目录：", PROCESSED_DATA_DIR.resolve())

项目根目录： C:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing
原始数据目录： C:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\data\raw
处理后数据目录： C:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\data\processed


In [20]:

TRAIN_RAW_PATH = RAW_DATA_DIR / "banking77_train.csv"
TEST_RAW_PATH = RAW_DATA_DIR / "banking77_test.csv"

print("训练集保存路径：", TRAIN_RAW_PATH)
print("测试集保存路径：", TEST_RAW_PATH)

训练集保存路径： c:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\data\raw\banking77_train.csv
测试集保存路径： c:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\data\raw\banking77_test.csv


In [21]:


EXPECTED_COLUMNS = {"text", "category"}


def load_banking77_csv(file_path: Path, source_split: str) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(f"文件不存在：{file_path}")

    try:
        dataframe = pd.read_csv(file_path)
    except Exception as error:
        raise RuntimeError(f"无法读取CSV文件：{file_path}") from error

    if dataframe.empty:
        raise ValueError(f"数据文件为空：{file_path}")

    missing_columns = EXPECTED_COLUMNS - set(dataframe.columns)

    if missing_columns:
        raise ValueError(
            f"{file_path.name}缺少必要列：{sorted(missing_columns)}；"
            f"实际列名：{dataframe.columns.tolist()}"
        )

    dataframe = dataframe[["text", "category"]].copy()
    dataframe["source_split"] = source_split

    return dataframe

In [22]:


original_train_df = load_banking77_csv(
    TRAIN_RAW_PATH,
    source_split="official_train"
)

original_test_df = load_banking77_csv(
    TEST_RAW_PATH,
    source_split="official_test"
)

print("官方训练集形状：", original_train_df.shape)
print("官方测试集形状：", original_test_df.shape)

display(original_train_df.head())

官方训练集形状： (10003, 3)
官方测试集形状： (3080, 3)


,text,category,source_split
0,I am still waiting on my card?,card_arrival,official_train
1,What can I do if my card still hasn't arrived ...,card_arrival,official_train
2,I have been waiting over a week. Is the card s...,card_arrival,official_train
3,Can I track my card while it is in the process...,card_arrival,official_train
4,"How do I know if I will get my card, or if it ...",card_arrival,official_train


In [23]:


print("训练集字段信息：")
original_train_df.info()

print("\n训练集类别数量：")
print(original_train_df["category"].nunique())

print("\n测试集类别数量：")
print(original_test_df["category"].nunique())

print("\n全部类别名称：")
all_categories = sorted(
    set(original_train_df["category"]).union(
        set(original_test_df["category"])
    )
)

for category in all_categories:
    print(category)

训练集字段信息：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10003 entries, 0 to 10002
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   text          10003 non-null  object
 1   category      10003 non-null  object
 2   source_split  10003 non-null  object
dtypes: object(3)
memory usage: 234.6+ KB

训练集类别数量：
77

测试集类别数量：
77

全部类别名称：
Refund_not_showing_up
activate_my_card
age_limit
apple_pay_or_google_pay
atm_support
automatic_top_up
balance_not_updated_after_bank_transfer
balance_not_updated_after_cheque_or_cash_deposit
beneficiary_not_allowed
cancel_transfer
card_about_to_expire
card_acceptance
card_arrival
card_delivery_estimate
card_linking
card_not_working
card_payment_fee_charged
card_payment_not_recognised
card_payment_wrong_exchange_rate
card_swallowed
cash_withdrawal_charge
cash_withdrawal_not_recognised
change_pin
compromised_card
contactless_not_working
country_support
declined_card_payment
declined_cash_wit

In [24]:

all_data_df = pd.concat(
    [original_train_df, original_test_df],
    axis=0,
    ignore_index=True
)

TARGET_INTENTS = sorted(all_data_df["category"].dropna().unique().tolist())

if len(TARGET_INTENTS) != 77:
    raise ValueError(
        f"预期BANKING77包含77个类别，实际发现{len(TARGET_INTENTS)}个类别。"
    )

print("合并后数据形状：", all_data_df.shape)
print("类别数量：", len(TARGET_INTENTS))
print("总样本数：", len(all_data_df))

合并后数据形状： (13083, 3)
类别数量： 77
总样本数： 13083


In [25]:
print("列名：", all_data_df.columns.tolist())
print("行数：", all_data_df.shape[0])
print("列数：", all_data_df.shape[1])

列名： ['text', 'category', 'source_split']
行数： 13083
列数： 3


处理缺失值

In [ ]:
# Cell 9：检查缺失值

missing_value_summary = all_data_df.isna().sum()

print("各字段缺失值数量：")
display(missing_value_summary.to_frame(name="missing_count"))

missing_text_count = int(all_data_df["text"].isna().sum())
missing_category_count = int(all_data_df["category"].isna().sum())

print("文本缺失数量：", missing_text_count)
print("标签缺失数量：", missing_category_count)

各字段缺失值数量：


,missing_count
text,0
category,0
source_split,0


文本缺失数量： 0
标签缺失数量： 0


In [27]:
# Cell 10：检查空文本和空标签

text_as_string = all_data_df["text"].fillna("").astype(str)
category_as_string = all_data_df["category"].fillna("").astype(str)

empty_text_mask = text_as_string.str.strip().eq("")
empty_category_mask = category_as_string.str.strip().eq("")

empty_text_count = int(empty_text_mask.sum())
empty_category_count = int(empty_category_mask.sum())

print("空文本数量：", empty_text_count)
print("空标签数量：", empty_category_count)

if empty_text_count > 0:
    display(all_data_df.loc[empty_text_mask].head(20))

if empty_category_count > 0:
    display(all_data_df.loc[empty_category_mask].head(20))

空文本数量： 0
空标签数量： 0


In [28]:
# Cell 11：删除缺失文本、缺失标签和空字符串

cleaned_data_df = all_data_df.dropna(
    subset=["text", "category"]
).copy()

cleaned_data_df["text"] = cleaned_data_df["text"].astype(str)
cleaned_data_df["category"] = cleaned_data_df["category"].astype(str)

cleaned_data_df = cleaned_data_df[
    cleaned_data_df["text"].str.strip().ne("")
    & cleaned_data_df["category"].str.strip().ne("")
].copy()

cleaned_data_df.reset_index(drop=True, inplace=True)

print("清理前样本数量：", len(all_data_df))
print("清理后样本数量：", len(cleaned_data_df))
print("清理后类别数量：", cleaned_data_df["category"].nunique())

if cleaned_data_df["category"].nunique() != 77:
    raise ValueError("数据清理后类别数量不是77，请检查数据。")

清理前样本数量： 13083
清理后样本数量： 13083
清理后类别数量： 77


标记1

In [29]:
# Cell 12：生成用于重复样本检查的标准化文本
# 这里只用于检查重复，不会覆盖原始text字段

import re


def normalize_text_for_duplicate_check(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


cleaned_data_df["normalized_text"] = cleaned_data_df["text"].apply(
    normalize_text_for_duplicate_check
)

display(
    cleaned_data_df[
        ["text", "normalized_text", "category"]
    ].head()
)

,text,normalized_text,category
0,I am still waiting on my card?,i am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,what can i do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,i have been waiting over a week. is the card s...,card_arrival
3,Can I track my card while it is in the process...,can i track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...","how do i know if i will get my card, or if it ...",card_arrival


In [30]:
# Cell 13：检查完全重复的文本和标签组合

duplicate_mask = cleaned_data_df.duplicated(
    subset=["normalized_text", "category"],
    keep=False
)

duplicate_samples_df = cleaned_data_df[
    duplicate_mask
].copy()

print("重复样本涉及的总行数：", len(duplicate_samples_df))

if not duplicate_samples_df.empty:
    display(
        duplicate_samples_df[
            ["text", "normalized_text", "category"]
        ]
        .sort_values(["normalized_text", "category"])
        .head(50)
    )

重复样本涉及的总行数： 24


,text,normalized_text,category
4576,\nAt which ATMs can I use this card?,at which atms can i use this card?,atm_support
11477,At which ATMs can I use this card?,at which atms can i use this card?,atm_support
6910,Do I need to go to a physical bank to change m...,do i need to go to a physical bank to change m...,change_pin
6965,\nDo I need to go to a physical bank to change...,do i need to go to a physical bank to change m...,change_pin
1722,\nHow do I unblock my PIN?,how do i unblock my pin?,pin_blocked
10557,How do I unblock my PIN?,how do i unblock my pin?,pin_blocked
1246,I can't seem to be able to use my card,i can't seem to be able to use my card,card_not_working
1290,\nI can't seem to be able to use my card\n\n\n,i can't seem to be able to use my card,card_not_working
9921,I don't live in the UK. Can I still get a card?,i don't live in the uk. can i still get a card?,country_support
13073,I don't live in the UK. Can I still get a card?,i don't live in the uk. can i still get a card?,country_support


In [31]:
# Cell 14：检查相同文本是否对应不同标签
# 如果相同文本对应多个标签，可能存在标签冲突

label_count_per_text = (
    cleaned_data_df
    .groupby("normalized_text")["category"]
    .nunique()
)

conflicting_texts = label_count_per_text[
    label_count_per_text > 1
].index

conflict_df = cleaned_data_df[
    cleaned_data_df["normalized_text"].isin(conflicting_texts)
].copy()

print("存在标签冲突的文本数量：", len(conflicting_texts))
print("标签冲突涉及的样本行数：", len(conflict_df))

if not conflict_df.empty:
    display(
        conflict_df[
            ["text", "normalized_text", "category"]
        ]
        .sort_values(["normalized_text", "category"])
        .head(100)
    )

存在标签冲突的文本数量： 0
标签冲突涉及的样本行数： 0


In [32]:
# Cell 15：如果存在标签冲突，先停止处理

if not conflict_df.empty:
    raise ValueError(
        "发现相同文本对应不同标签的情况。"
        "请先检查上方输出，再决定如何处理。"
    )

print("未发现相同文本对应不同标签的问题。")

未发现相同文本对应不同标签的问题。


In [33]:
# Cell 16：在数据划分前删除重复样本
# 防止相同文本进入训练集和测试集造成数据泄漏

deduplicated_data_df = cleaned_data_df.drop_duplicates(
    subset=["normalized_text", "category"],
    keep="first"
).copy()

deduplicated_data_df.reset_index(drop=True, inplace=True)

removed_duplicate_count = (
    len(cleaned_data_df) - len(deduplicated_data_df)
)

print("去重前样本数量：", len(cleaned_data_df))
print("去重后样本数量：", len(deduplicated_data_df))
print("删除的重复样本数量：", removed_duplicate_count)
print("去重后类别数量：", deduplicated_data_df["category"].nunique())

if deduplicated_data_df["category"].nunique() != 77:
    raise ValueError("去重后类别数量不是77，请检查重复样本处理。")

去重前样本数量： 13083
去重后样本数量： 13071
删除的重复样本数量： 12
去重后类别数量： 77


In [34]:
# Cell 17：准备划分数据
# source_split和normalized_text只用于审计，不作为模型输入

data_for_split = deduplicated_data_df[
    ["text", "category", "normalized_text"]
].copy()

print("待划分数据形状：", data_for_split.shape)
display(data_for_split.head())

待划分数据形状： (13071, 3)


,text,category,normalized_text
0,I am still waiting on my card?,card_arrival,i am still waiting on my card?
1,What can I do if my card still hasn't arrived ...,card_arrival,what can i do if my card still hasn't arrived ...
2,I have been waiting over a week. Is the card s...,card_arrival,i have been waiting over a week. is the card s...
3,Can I track my card while it is in the process...,card_arrival,can i track my card while it is in the process...
4,"How do I know if I will get my card, or if it ...",card_arrival,"how do i know if i will get my card, or if it ..."


疑问

In [35]:
# Cell 18：第一次分层划分
# 70%训练集，30%临时数据集

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

train_df, temporary_df = train_test_split(
    data_for_split,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=data_for_split["category"]
)

print("训练集样本数量：", len(train_df))
print("临时数据集样本数量：", len(temporary_df))

训练集样本数量： 9149
临时数据集样本数量： 3922


In [36]:
# Cell 19：第二次分层划分
# 将30%的临时数据平均分成15%验证集和15%测试集

validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temporary_df["category"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("训练集样本数量：", len(train_df))
print("验证集样本数量：", len(validation_df))
print("测试集样本数量：", len(test_df))

print(
    "划分后总样本数量：",
    len(train_df) + len(validation_df) + len(test_df)
)

训练集样本数量： 9149
验证集样本数量： 1961
测试集样本数量： 1961
划分后总样本数量： 13071


In [37]:
# Cell 20：检查实际划分比例

total_sample_count = len(data_for_split)

split_summary_df = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "sample_count": [
            len(train_df),
            len(validation_df),
            len(test_df)
        ],
        "actual_ratio": [
            len(train_df) / total_sample_count,
            len(validation_df) / total_sample_count,
            len(test_df) / total_sample_count
        ]
    }
)

display(split_summary_df)

,split,sample_count,actual_ratio
0,train,9149,0.699946
1,validation,1961,0.150027
2,test,1961,0.150027


In [38]:
# Cell 21：检查三个数据集是否都包含全部77类

split_dataframes = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

for split_name, dataframe in split_dataframes.items():
    class_count = dataframe["category"].nunique()

    print(f"{split_name}类别数量：{class_count}")

    if class_count != 77:
        raise ValueError(
            f"{split_name}数据集应包含77类，"
            f"实际包含{class_count}类。"
        )

print("训练集、验证集和测试集均包含全部77个类别。")

train类别数量：77
validation类别数量：77
test类别数量：77
训练集、验证集和测试集均包含全部77个类别。


In [39]:
# Cell 22：查看三个数据集的类别分布

def get_class_distribution(
    dataframe: pd.DataFrame,
    split_name: str
) -> pd.DataFrame:

    distribution = (
        dataframe["category"]
        .value_counts()
        .rename_axis("category")
        .reset_index(name="sample_count")
    )

    distribution["sample_ratio"] = (
        distribution["sample_count"] / len(dataframe)
    )

    distribution["split"] = split_name

    return distribution


train_distribution = get_class_distribution(
    train_df,
    "train"
)

validation_distribution = get_class_distribution(
    validation_df,
    "validation"
)

test_distribution = get_class_distribution(
    test_df,
    "test"
)

all_distributions_df = pd.concat(
    [
        train_distribution,
        validation_distribution,
        test_distribution
    ],
    ignore_index=True
)

distribution_table = all_distributions_df.pivot(
    index="category",
    columns="split",
    values="sample_count"
)

display(distribution_table)

split,test,train,validation
category,,,
Refund_not_showing_up,30,141,31
activate_my_card,30,139,30
age_limit,22,105,23
apple_pay_or_google_pay,25,116,25
atm_support,19,87,18
...,...,...,...
virtual_card_not_working,12,57,12
visa_or_mastercard,26,122,27
why_verify_identity,24,113,24


In [40]:
# Cell 23：检查训练集、验证集和测试集之间是否存在文本重叠

train_texts = set(train_df["normalized_text"])
validation_texts = set(validation_df["normalized_text"])
test_texts = set(test_df["normalized_text"])

train_validation_overlap = train_texts.intersection(
    validation_texts
)

train_test_overlap = train_texts.intersection(
    test_texts
)

validation_test_overlap = validation_texts.intersection(
    test_texts
)

print(
    "训练集与验证集重复文本数量：",
    len(train_validation_overlap)
)

print(
    "训练集与测试集重复文本数量：",
    len(train_test_overlap)
)

print(
    "验证集与测试集重复文本数量：",
    len(validation_test_overlap)
)

if (
    train_validation_overlap
    or train_test_overlap
    or validation_test_overlap
):
    raise ValueError(
        "不同数据集之间存在重复文本，可能造成数据泄漏。"
    )

print("三个数据集之间不存在重复文本。")

训练集与验证集重复文本数量： 0
训练集与测试集重复文本数量： 0
验证集与测试集重复文本数量： 0
三个数据集之间不存在重复文本。


In [41]:
# Cell 24：生成最终用于模型训练的数据
# 模型只需要text和category两列

final_train_df = train_df[
    ["text", "category"]
].copy()

final_validation_df = validation_df[
    ["text", "category"]
].copy()

final_test_df = test_df[
    ["text", "category"]
].copy()

print("最终训练集：", final_train_df.shape)
print("最终验证集：", final_validation_df.shape)
print("最终测试集：", final_test_df.shape)

display(final_train_df.head())

最终训练集： (9149, 2)
最终验证集： (1961, 2)
最终测试集： (1961, 2)


,text,category
0,I want to get Visa and Mastercard,visa_or_mastercard
1,Why is identity verification required?,why_verify_identity
2,I tried using my ATM card at your Notting Hill...,declined_cash_withdrawal
3,What do I do with my card PIN?,get_physical_card
4,I can't find the top-up verification code.,verify_top_up


In [42]:
# Cell 25：创建处理后数据目录

from pathlib import Path

if "PROCESSED_DATA_DIR" not in globals():
    CURRENT_DIR = Path.cwd()

    if CURRENT_DIR.name == "notebooks":
        PROJECT_ROOT = CURRENT_DIR.parent
    else:
        PROJECT_ROOT = CURRENT_DIR

    PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("处理后数据目录：", PROCESSED_DATA_DIR.resolve())

处理后数据目录： C:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\data\processed


In [ ]:
# Cell 26：保存训练集、验证集和测试集

TRAIN_SAVE_PATH = PROCESSED_DATA_DIR / "train_77.csv"
VALIDATION_SAVE_PATH = PROCESSED_DATA_DIR / "validation_77.csv"
TEST_SAVE_PATH = PROCESSED_DATA_DIR / "test_77.csv"

final_train_df.to_csv(
    TRAIN_SAVE_PATH,
    index=False,
    encoding="utf-8-sig"
)

final_validation_df.to_csv(
    VALIDATION_SAVE_PATH,
    index=False,
    encoding="utf-8-sig"
)

final_test_df.to_csv(
    TEST_SAVE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("训练集保存路径：", TRAIN_SAVE_PATH)
print("验证集保存路径：", VALIDATION_SAVE_PATH)
print("测试集保存路径：", TEST_SAVE_PATH)

训练集保存路径： c:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\data\processed\train_77.csv
验证集保存路径： c:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\data\processed\validation_77.csv
测试集保存路径： c:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\data\processed\test_77.csv


In [44]:
# Cell 27：重新读取保存后的文件，确认文件可以正常使用

saved_train_df = pd.read_csv(TRAIN_SAVE_PATH)
saved_validation_df = pd.read_csv(VALIDATION_SAVE_PATH)
saved_test_df = pd.read_csv(TEST_SAVE_PATH)

print("训练集形状：", saved_train_df.shape)
print("验证集形状：", saved_validation_df.shape)
print("测试集形状：", saved_test_df.shape)

display(saved_train_df.head())

训练集形状： (9149, 2)
验证集形状： (1961, 2)
测试集形状： (1961, 2)


,text,category
0,I want to get Visa and Mastercard,visa_or_mastercard
1,Why is identity verification required?,why_verify_identity
2,I tried using my ATM card at your Notting Hill...,declined_cash_withdrawal
3,What do I do with my card PIN?,get_physical_card
4,I can't find the top-up verification code.,verify_top_up


In [45]:
# Cell 28：检查保存后的字段、空数据和类别数量

expected_columns = ["text", "category"]

saved_datasets = {
    "train": saved_train_df,
    "validation": saved_validation_df,
    "test": saved_test_df,
}

for split_name, dataframe in saved_datasets.items():

    if dataframe.empty:
        raise ValueError(f"{split_name}数据集为空。")

    if dataframe.columns.tolist() != expected_columns:
        raise ValueError(
            f"{split_name}字段错误。"
            f"预期字段：{expected_columns}，"
            f"实际字段：{dataframe.columns.tolist()}"
        )

    if dataframe["text"].isna().any():
        raise ValueError(f"{split_name}中存在缺失文本。")

    if dataframe["category"].isna().any():
        raise ValueError(f"{split_name}中存在缺失标签。")

    class_count = dataframe["category"].nunique()

    if class_count != 77:
        raise ValueError(
            f"{split_name}类别数量错误，"
            f"预期77类，实际{class_count}类。"
        )

    print(
        f"{split_name}检查通过："
        f"{len(dataframe)}条样本，{class_count}个类别"
    )

train检查通过：9149条样本，77个类别
validation检查通过：1961条样本，77个类别
test检查通过：1961条样本，77个类别


In [46]:
# Cell 29：检查三个数据集的样本总数和实际比例

total_saved_samples = (
    len(saved_train_df)
    + len(saved_validation_df)
    + len(saved_test_df)
)

split_ratio_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "sample_count": [
            len(saved_train_df),
            len(saved_validation_df),
            len(saved_test_df),
        ],
        "sample_ratio": [
            len(saved_train_df) / total_saved_samples,
            len(saved_validation_df) / total_saved_samples,
            len(saved_test_df) / total_saved_samples,
        ],
    }
)

print("保存后的总样本数量：", total_saved_samples)

display(split_ratio_summary)

保存后的总样本数量： 13071


,split,sample_count,sample_ratio
0,train,9149,0.699946
1,validation,1961,0.150027
2,test,1961,0.150027


In [47]:
# Cell 30：再次检查三个数据集之间是否存在重复文本

import re


def normalize_text_for_check(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


train_text_set = set(
    saved_train_df["text"].apply(normalize_text_for_check)
)

validation_text_set = set(
    saved_validation_df["text"].apply(normalize_text_for_check)
)

test_text_set = set(
    saved_test_df["text"].apply(normalize_text_for_check)
)

train_validation_overlap = train_text_set.intersection(
    validation_text_set
)

train_test_overlap = train_text_set.intersection(
    test_text_set
)

validation_test_overlap = validation_text_set.intersection(
    test_text_set
)

print(
    "训练集与验证集重复文本数量：",
    len(train_validation_overlap)
)

print(
    "训练集与测试集重复文本数量：",
    len(train_test_overlap)
)

print(
    "验证集与测试集重复文本数量：",
    len(validation_test_overlap)
)

if (
    train_validation_overlap
    or train_test_overlap
    or validation_test_overlap
):
    raise ValueError(
        "不同数据集之间存在重复文本，可能发生数据泄漏。"
    )

print("数据集之间不存在重复文本。")

训练集与验证集重复文本数量： 0
训练集与测试集重复文本数量： 0
验证集与测试集重复文本数量： 0
数据集之间不存在重复文本。


In [48]:
# Cell 31：查看三个数据集的类别分布

def calculate_class_distribution(
    dataframe: pd.DataFrame,
    split_name: str
) -> pd.DataFrame:

    distribution = (
        dataframe["category"]
        .value_counts()
        .rename_axis("category")
        .reset_index(name="sample_count")
    )

    distribution["sample_ratio"] = (
        distribution["sample_count"] / len(dataframe)
    )

    distribution["split"] = split_name

    return distribution


train_distribution = calculate_class_distribution(
    saved_train_df,
    "train"
)

validation_distribution = calculate_class_distribution(
    saved_validation_df,
    "validation"
)

test_distribution = calculate_class_distribution(
    saved_test_df,
    "test"
)

all_distribution_df = pd.concat(
    [
        train_distribution,
        validation_distribution,
        test_distribution,
    ],
    ignore_index=True
)

class_count_table = all_distribution_df.pivot(
    index="category",
    columns="split",
    values="sample_count"
)

display(class_count_table)

split,test,train,validation
category,,,
Refund_not_showing_up,30,141,31
activate_my_card,30,139,30
age_limit,22,105,23
apple_pay_or_google_pay,25,116,25
atm_support,19,87,18
...,...,...,...
virtual_card_not_working,12,57,12
visa_or_mastercard,26,122,27
why_verify_identity,24,113,24


In [49]:
# Cell 32：保存类别分布报告

REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_DISTRIBUTION_SAVE_PATH = (
    REPORT_DIR / "class_distribution_77.csv"
)

all_distribution_df.to_csv(
    CLASS_DISTRIBUTION_SAVE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "类别分布报告保存路径：",
    CLASS_DISTRIBUTION_SAVE_PATH
)

类别分布报告保存路径： c:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\reports\class_distribution_77.csv


In [50]:
# Cell 33：建立类别名称与数字编号之间的映射
# 后续模型通常需要将字符串标签转换为数字标签

import json


all_labels = sorted(
    saved_train_df["category"].unique().tolist()
)

if len(all_labels) != 77:
    raise ValueError(
        f"标签数量错误，预期77，实际{len(all_labels)}。"
    )

label_to_id = {
    label: index
    for index, label in enumerate(all_labels)
}

id_to_label = {
    index: label
    for label, index in label_to_id.items()
}

print("类别数量：", len(label_to_id))

for label, label_id in list(label_to_id.items())[:10]:
    print(f"{label_id}: {label}")

类别数量： 77
0: Refund_not_showing_up
1: activate_my_card
2: age_limit
3: apple_pay_or_google_pay
4: atm_support
5: automatic_top_up
6: balance_not_updated_after_bank_transfer
7: balance_not_updated_after_cheque_or_cash_deposit
8: beneficiary_not_allowed
9: cancel_transfer


In [51]:
# Cell 34：保存类别编号映射

LABEL_TO_ID_PATH = (
    PROCESSED_DATA_DIR / "label_to_id.json"
)

ID_TO_LABEL_PATH = (
    PROCESSED_DATA_DIR / "id_to_label.json"
)

with open(
    LABEL_TO_ID_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        label_to_id,
        file,
        ensure_ascii=False,
        indent=4
    )

with open(
    ID_TO_LABEL_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        id_to_label,
        file,
        ensure_ascii=False,
        indent=4
    )

print("label_to_id保存路径：", LABEL_TO_ID_PATH)
print("id_to_label保存路径：", ID_TO_LABEL_PATH)

label_to_id保存路径： c:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\data\processed\label_to_id.json
id_to_label保存路径： c:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\data\processed\id_to_label.json


In [52]:
# Cell 35：阶段0最终状态汇总

stage_zero_summary = pd.DataFrame(
    {
        "item": [
            "训练集样本数",
            "验证集样本数",
            "测试集样本数",
            "总样本数",
            "类别数量",
            "训练-验证重复文本",
            "训练-测试重复文本",
            "验证-测试重复文本",
        ],
        "value": [
            len(saved_train_df),
            len(saved_validation_df),
            len(saved_test_df),
            total_saved_samples,
            saved_train_df["category"].nunique(),
            len(train_validation_overlap),
            len(train_test_overlap),
            len(validation_test_overlap),
        ],
    }
)

display(stage_zero_summary)

print("阶段0完成：")
print("1. BANKING77全部77类数据已加载")
print("2. 缺失值、空文本和重复文本已处理")
print("3. 数据已按70%/15%/15%分层划分")
print("4. 数据集之间未发现重复文本")
print("5. 训练集、验证集和测试集已保存")
print("6. 类别名称与数字编号映射已保存")
print("7. 当前尚未进行模型训练")

,item,value
0,训练集样本数,9149
1,验证集样本数,1961
2,测试集样本数,1961
3,总样本数,13071
4,类别数量,77
5,训练-验证重复文本,0
6,训练-测试重复文本,0
7,验证-测试重复文本,0


阶段0完成：
1. BANKING77全部77类数据已加载
2. 缺失值、空文本和重复文本已处理
3. 数据已按70%/15%/15%分层划分
4. 数据集之间未发现重复文本
5. 训练集、验证集和测试集已保存
6. 类别名称与数字编号映射已保存
7. 当前尚未进行模型训练
